# HydroSense-Kenya - Level 2: NumPy, Vectorization, Floating Point Errors, and Numerical Reliability
**Course:** ICS 2207 Scientific Computing  
**Level:** 2 of 6 (15 marks)

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os

os.makedirs('../outputs', exist_ok=True)

# Load datasets
weather = pd.read_csv('../data/raw/weather_daily.csv', na_values=['NA', ''])
soil    = pd.read_csv('../data/raw/soil_sensor_data.csv', na_values=['NA', ''])
params  = pd.read_csv('../data/raw/crop_zone_parameters.csv')

# Fill missing values with column means
weather['rainfall_mm']   = weather['rainfall_mm'].fillna(weather['rainfall_mm'].mean())
weather['temperature_c'] = weather['temperature_c'].fillna(weather['temperature_c'].mean())
weather['humidity_pct']  = weather['humidity_pct'].fillna(weather['humidity_pct'].mean())

# Extract arrays for computation
T     = weather['temperature_c'].values
W     = weather['wind_speed_mps'].values
Solar = weather['solar_index'].values
H     = weather['humidity_pct'].values

print(f'Dataset ready: {len(T)} days of weather data.')

## 2. Python Loop vs NumPy Vectorization

We compute daily ET two ways - first with an explicit Python loop, then with NumPy array operations - and compare their speed and correctness.

In [ ]:
# ─ Method 1: Python for-loop ─
N_REPEATS = 10000  # Repeat to get measurable timing

start_loop = time.time()
for _ in range(N_REPEATS):
    et_loop = []
    for i in range(len(T)):
        val = 0.12*T[i] + 0.35*W[i] + 2.4*Solar[i] - 0.025*H[i]
        et_loop.append(max(0.0, val))
    et_loop = np.array(et_loop)
time_loop = (time.time() - start_loop) / N_REPEATS * 1000  # ms per call

# ─ Method 2: NumPy vectorized ─
start_vec = time.time()
for _ in range(N_REPEATS):
    et_vec = np.maximum(0.0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)
time_vec = (time.time() - start_vec) / N_REPEATS * 1000  # ms per call

speedup = time_loop / time_vec

print('=== Timing Results ===')
print(f'Python loop time:     {time_loop:.4f} ms per call')
print(f'NumPy vectorized:     {time_vec:.4f} ms per call')
print(f'Speedup factor:       {speedup:.1f}x faster with NumPy')
print(f'Results identical:    {np.allclose(et_loop, et_vec)}')

In [ ]:
# ─Comparison Table (Markdown) ─
print('| Method            | Time (ms)     | Speedup | Code complexity |')
print('|-------------------|---------------|---------|-----------------|')
print(f'| Python for-loop   | {time_loop:.4f}      | 1x      | 4 lines         |')
print(f'| NumPy vectorized  | {time_vec:.4f}      | {speedup:.0f}x      | 1 line          |')

### Why is NumPy faster?

Python loops interpret and dispatch each operation one at a time through the Python interpreter - slow because of the overhead of type-checking and object management on every iteration. NumPy operations are executed in pre-compiled C code that processes the entire array in a single call, avoiding interpreter overhead entirely. For a 30-element array the speedup is already significant; for arrays of millions of sensor readings, vectorization is not optional - it is essential.

## 3. Floating Point Behaviour and Limitations

In [ ]:
print('=== Floating Point Demonstration ===\n')

# Classic example: 0.1 + 0.2 in binary floating point
a = 0.1
b = 0.2
c = a + b
print(f'0.1 + 0.2 = {c}')                        # Should be 0.3
print(f'Exact representation: {c:.20f}')           # Shows the hidden digits
print(f'0.1 + 0.2 == 0.3:     {c == 0.3}')        # False - never use == for floats
print(f'np.isclose(0.1+0.2, 0.3): {np.isclose(c, 0.3)}')  # Correct approach

print()
print('Why? Computers store numbers in binary (base 2).')
print('0.1 in binary is 0.000110011001100... (infinite repeating)')
print('Just like 1/3 in decimal = 0.333... never terminates.')
print('When truncated to 64 bits, a tiny rounding error is introduced.')

In [ ]:
print('=== Accumulation of Rounding Error over 30-Day Simulation ===\n')

# Scenario: Small daily ET = 0.1 mm/day subtracted from soil moisture
# After 30 days, we expect: 30.0 - 30*0.1 = 27.0%
S_loop = 30.0
for day in range(30):
    S_loop -= 0.1

S_expected = 30.0 - 30 * 0.1

print(f'Expected S after 30 days:  {S_expected:.10f}%')
print(f'Loop result after 30 days: {S_loop:.10f}%')
print(f'Difference (error):        {abs(S_loop - S_expected):.2e}%')
print(f'Are they close enough?     {np.isclose(S_loop, S_expected)}')
print()
print('The error is tiny (~1e-15) but would accumulate in a year-long simulation.')
print('Fix: use numpy cumulative operations instead of loops where possible.')

In [ ]:
print('=== Integer vs Float Division ===\n')
print(f'5 / 2  = {5/2}    (true division - always float in Python 3)')
print(f'5 // 2 = {5//2}     (floor division - rounds down to integer)')
print(f'5 % 2  = {5%2}     (modulus - remainder)')
print()
print('=== Comparing Floats Correctly ===')
et_a = 0.12*23.8 + 0.35*2.28 + 2.4*0.78 - 0.025*69.7
et_b = compute_et = max(0, 0.12*23.8 + 0.35*2.28 + 2.4*0.78 - 0.025*69.7)
print(f'et_a == et_b:              {et_a == et_b}   (exact comparison - risky)')
print(f'np.isclose(et_a, et_b):    {np.isclose(et_a, et_b)}  (safe comparison)')

## 4. Error Propagation - How Sensor Noise Affects Decisions

In [ ]:
np.random.seed(42)  # Fix seed for reproducibility

# Baseline ET (no noise)
et_baseline = np.maximum(0.0, 0.12*T + 0.35*W + 2.4*Solar - 0.025*H)

# Simulate increasing noise levels on the temperature sensor
noise_levels = [0.0, 0.01, 0.05, 0.10, 0.20]  # 0% to 20% noise
et_results   = {}

for noise in noise_levels:
    T_noisy = T * (1 + np.random.normal(0, noise, size=len(T)))
    et_noisy = np.maximum(0.0, 0.12*T_noisy + 0.35*W + 2.4*Solar - 0.025*H)
    et_results[noise] = et_noisy

# How much does irrigation recommendation shift?
# Simple irrigation = max(0, target - (S + rainfall - ET))
# ET error directly shifts irrigation recommendations
target_zone_a = 33.0
S_day1        = 33.2
rain_day1     = 3.2

print('=== Impact of ET Error on Day-1 Irrigation Recommendation for Zone A ===')
print(f'{"Noise Level":<15} {"ET (mm)":<12} {"Irr. Needed (mm)":<20} {"Error vs Baseline"}')
print('-' * 65)
for noise in noise_levels:
    et_val = et_results[noise][0]
    irr    = max(0, target_zone_a - (S_day1 + rain_day1 - et_val))
    irr_base = max(0, target_zone_a - (S_day1 + rain_day1 - et_baseline[0]))
    print(f'{int(noise*100):>3}% noise      {et_val:<12.4f} {irr:<20.4f} {irr - irr_base:+.4f}')

In [ ]:
# ── Visualisation ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

days = range(len(T))

# Left: ET estimates under different noise levels
for noise in noise_levels:
    label = f'{int(noise*100)}% noise'
    lw    = 2 if noise == 0 else 1
    axes[0].plot(days, et_results[noise], label=label, linewidth=lw, alpha=0.8)

axes[0].set_xlabel('Day of 30th March 2026')
axes[0].set_ylabel('ET Estimate (mm/day)')
axes[0].set_title('Effect of Temperature Sensor Noise on ET Estimates')
axes[0].legend(fontsize=9)

# Right: Standard deviation of ET error at each noise level
et_stds = [np.std(et_results[n] - et_baseline) for n in noise_levels]
bars = axes[1].bar([f'{int(n*100)}%' for n in noise_levels], et_stds,
                   color=['#2ecc71','#3498db','#f39c12','#e67e22','#e74c3c'])
axes[1].set_xlabel('Temperature Sensor Noise Level')
axes[1].set_ylabel('Std Dev of ET Error (mm/day)')
axes[1].set_title('Sensor Noise Level vs Prediction Error')
for bar, val in zip(bars, et_stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/level2_error_propagation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

### Error Propagation Interpretation

A temperature sensor with **5% noise** introduces an ET prediction error of approximately 0.15 mm/day on average. Since ET appears in the water balance equation daily, this error accumulates over 30 days to ~4.5 mm of cumulative moisture error - equivalent to roughly a full day's irrigation being misallocated. At **10% sensor noise**, this doubles. In a real farm context, this translates directly to wasted water or avoidable crop stress, making sensor calibration and quality control (Level 4) a critical step before any modelling.

## 5. NumPy Broadcasting and Vectorized Water Balance

In [ ]:
# Vectorized water balance for all 3 zones simultaneously using NumPy broadcasting

# Zone parameters as NumPy arrays
targets      = params['target_moisture_pct'].values   # [33, 35, 31]
mins         = params['min_moisture_pct'].values       # [22, 24, 20]
field_caps   = params['field_capacity_pct'].values     # [41, 43, 40]
drain_coeffs = params['drainage_coefficient'].values   # [0.18, 0.15, 0.22]

# Starting moisture from day 1 soil readings
S = np.array([
    soil[soil['zone_id'] == 'Zone_A']['soil_moisture_pct'].iloc[0],
    soil[soil['zone_id'] == 'Zone_B']['soil_moisture_pct'].iloc[0],
    soil[soil['zone_id'] == 'Zone_C']['soil_moisture_pct'].iloc[0]
])
print(f'Starting moisture: Zone A={S[0]}%, Zone B={S[1]}%, Zone C={S[2]}%')

# Simulate one step: day 1
R      = np.array([3.2, 3.2, 3.2])   # same rainfall all zones
I      = np.array([0.0, 0.0, 0.0])   # no irrigation
ET     = et_baseline[0]

# Vectorized: compute drainage and next state for all zones at once
drainage = drain_coeffs * np.maximum(0, S - field_caps)
S_next   = np.clip(S + R + I - ET - drainage, 0, field_caps)

print(f'Next-day moisture: Zone A={S_next[0]:.2f}%, Zone B={S_next[1]:.2f}%, Zone C={S_next[2]:.2f}%')

# Stress check using boolean arrays (vectorized)
stressed = S_next < mins
for i, zone in enumerate(['Zone_A', 'Zone_B', 'Zone_C']):
    status = 'STRESSED' if stressed[i] else 'OK'
    print(f'  {zone}: {S_next[i]:.2f}% (min={mins[i]}%) → {status}')

## 6. Numerical Reliability - Best Practices Summary

### Why Floating Point Matters in Scientific Computing

Computers represent real numbers in IEEE 754 double-precision binary format using 64 bits. Because base-2 fractions cannot exactly represent all base-10 decimals (just as 1/3 cannot be written exactly in base 10), tiny rounding errors are introduced at every arithmetic operation.

In a 30-day soil moisture simulation, each daily subtraction of ET introduces a rounding error on the order of 10⁻¹⁵. While this is negligible for a single step, over a year-long simulation with multiple zones, these errors can accumulate to a physically meaningful level, affecting irrigation recommendations.

### Rules We Follow in HydroSense-Kenya

| Rule | Wrong | Right |
|------|-------|-------|
| Compare floats | `x == 0.3` | `np.isclose(x, 0.3)` |
| Batch computation | Python for-loop | NumPy vectorized operations |
| Prevent impossible values | No guard | `np.clip(S, 0, field_capacity)` |
| Reproducibility | No seed | `np.random.seed(42)` |
| Display | `print(x)` | `print(f'{x:.4f}')` with appropriate precision |

### Practical Impact

A 10% noise level on the temperature sensor shifts ET estimates by ~0.3 mm/day standard deviation. Compounded across 30 days across 3 zones, this creates up to 27 mm of cumulative irrigation error - equivalent to a full pump session wasted or a critical deficit missed. This motivates the data quality work in Level 4 and the uncertainty quantification (Monte Carlo) in Level 5.

## 7. Summary

- **Vectorization:** NumPy is significantly faster than Python loops. The speedup scales with array size - essential for large sensor datasets.
- **Floating point:** Binary representation causes unavoidable rounding errors. We must use `np.isclose()` for comparisons and `np.clip()` to keep values in valid physical ranges.
- **Error propagation:** Even small sensor calibration errors (5–10%) produce meaningful shifts in irrigation recommendations when accumulated over a full growing season.
- **Best practice:** Always set `np.random.seed()` for reproducibility, use vectorized operations, and validate results against known analytical values.

**Next step:** Level 3 implements the numerical methods engine - root finding, differentiation, integration, and linear systems - from scratch.